# 02 — Weather-closure windows per night

Two independent sources (handoff §1b, §5; `data/raw/weather_criteria_table.md`;
`data/raw/handoff_dynalene_condensation_criteria_2025-12_2026-01.md`):

1. **Observatory Status WEATHER bit** via `rubin_nights.observatory_status`, from 2026-02-25. Primary where it exists.
2. **EFD operating criteria** on the 1-minute cache written by `python/pull_efd_weather_1min.py`: wind (10-min mean > 20 m/s or
   sustained gust ≥ 25 m/s, 15-min reopen), outside temperature − dew point < 2 °C for 20 min, and the Dynalene − dew point
   condensation rule with the threshold valid on each `day_obs` (none before 2025-12-15; ≤ +2 °C to 2025-12-29; < +2 °C to
   2026-01-18; ≤ +1 °C on 2026-01-19; < −2 °C from 2026-01-20). Primary before 2026-02-25, cross-check afterwards.

Dynalene sensor: `MTMount.dynaleneCooling.dynaleneTemperatureAzimuth0001` (it reproduces the "Dynalene at Az" values quoted in the
night reports and sits within 0.3 °C of the handoff's `coolPipeSplyTemp − 1.5`; `Pier0102` is +1.5 °C warmer and is not used).

**Cloud is not in the EFD.** `closed_unexplained_hours` counts minutes with the shutter closed, or the telescope at horizon with
the mirror covers on, while no criterion fires. Those hours are handled in notebook 04.

In [ ]:
import sys, logging
from pathlib import Path
import numpy as np, pandas as pd
sys.path.insert(0, "..")
from time_loss import DAY_OBS_MIN, DAY_OBS_MAX, GAP_MIN_MINUTES, GAP_MIN_SENSITIVITY, DAY_OBS_WEATHER_BIT_START
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 300); pd.set_option("display.max_colwidth", 80)
logging.basicConfig(level=logging.WARNING)
for name in ("httpx", "numexpr", "time_loss.access"):
    logging.getLogger(name).setLevel(logging.WARNING)
STEP0 = Path("../data/step0"); CACHE = Path("../data/cache"); PROC = Path("../data/processed")
for p in (STEP0, CACHE, PROC): p.mkdir(parents=True, exist_ok=True)

In [ ]:
from time_loss import weather as W
limits = pd.read_csv(STEP0 / "night_limits.csv")
cache = pd.read_parquet(CACHE / "efd_weather_1min.parquet")   # python/pull_efd_weather_1min.py
periods = pd.read_parquet(STEP0 / "obs_state_periods.parquet")  # notebook 00
print("EFD cache:", len(cache), "minutes,", cache.day_obs.nunique(), "nights; NaN fraction per channel:")
print(cache.drop(columns="day_obs").isna().mean().round(3).to_dict())

In [ ]:
efd_tab, efd_res = W.efd_weather_table(cache, limits)
bit_tab, bit_res = W.weather_bit_table(periods, limits)
wx = W.combine_weather(bit_tab, efd_tab)
wx.to_csv(STEP0 / "weather_windows_per_night.csv", index=False)
# window lists (UTC MJD) for the accounting notebook
import pickle
with open(CACHE / "weather_windows.pkl", "wb") as fh:
    pickle.dump({"bit": bit_res, "efd": {d: r.windows for d, r in efd_res.items()}}, fh)
wx[["bit_weather_hours", "efd_weather_hours", "efd_wind_hours", "efd_dewpoint_hours", "efd_dynalene_hours", "closed_unexplained_hours"]].sum().round(1)

## WEATHER bit vs EFD criteria where both exist

In [ ]:
both = wx[wx.bit_available]
print(f"nights {len(both)} | bit {both.bit_weather_hours.sum():.1f} h | EFD criteria {both.efd_weather_hours.sum():.1f} h")
print("bit >= 2 h and EFD < 0.5 h (cloud closures):", both[(both.bit_weather_hours >= 2) & (both.efd_weather_hours < 0.5)].day_obs.tolist())
print("EFD >= 2 h and bit < 0.5 h:", both[(both.efd_weather_hours >= 2) & (both.bit_weather_hours < 0.5)].day_obs.tolist())

## Condensation era, Dec 2025 – Jan 2026 (own category per the condensation handoff)

In [ ]:
cols = ["day_obs", "efd_weather_hours", "efd_dynalene_hours", "efd_dewpoint_hours", "efd_wind_hours", "closed_unexplained_hours", "efd_minutes_missing"]
wx[(wx.day_obs >= 20251215) & (wx.day_obs <= 20260131)][cols]

## One night in detail

In [ ]:
DAY = 20260111  # 6.4 h lost to dew-point fluctuations per the night report
g = cache[cache.day_obs == DAY].drop(columns="day_obs")
f = W.flag_minutes(g, DAY)
import matplotlib.pyplot as plt
from time_loss.style import COLORS
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(g.index, g.dyn_az1 - g.dp_out, color=COLORS["weather"], label="Dynalene(Az1) − outside dew point")
axs[0].plot(g.index, g.t_out - g.dp_out, color=COLORS["nonweather_loss"], label="ambient − dew point")
axs[0].axhline(2, color=COLORS["grid"]); axs[0].legend(frameon=False); axs[0].set_ylabel("°C")
axs[1].plot(g.index, g.shutter, color=COLORS["ink"], label="shutter %"); axs[1].plot(g.index, g.mcover, color=COLORS["muted"], label="mirror cover pos"); axs[1].plot(g.index, g.el, color=COLORS["on_sky"], label="elevation"); axs[1].legend(frameon=False)
axs[2].fill_between(f.index, 0, f["dynalene"].astype(int), color=COLORS["weather"], step="mid", label="dynalene CLOSE")
axs[2].fill_between(f.index, 1, 1 + f["dewpoint"].astype(int), color=COLORS["nonweather_loss"], step="mid", label="dew-point CLOSE")
axs[2].set_yticks([]); axs[2].legend(frameon=False); fig.suptitle(f"day_obs {DAY}"); fig.tight_layout()